# Synthetic Datasets: Privacy Metrics (DCR & NNDR)

Author: Ilse Harmers \
Last modified: March 19, 2025

In [ ]:
# Importing libraries.
import numpy as np
np.seterr(divide = "ignore", invalid = "ignore")
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.metrics import pairwise_distances

## Initialization

In [ ]:
# Initializing the real datasets (Adult, Bank Marketing and Credit Card Default) and our functions. 

# Adult dataset.
adult_train = pd.read_csv("../train-test-datasets/adult/train_encoded.csv")
# Scaling the "education-num" column in Adult.
adult_train["education-num"] = ((adult_train["education-num"] - adult_train["education-num"].min()) / 
                                (adult_train["education-num"].max() - adult_train["education-num"].min()))

# Bank Marketing dataset.
bank_train = pd.read_csv("../train-test-datasets/bank-marketing/train_encoded.csv")
# Scaling the "day" column in Bank.
bank_train["day"] = ((bank_train["day"] - bank_train["day"].min()) / (bank_train["day"].max() - bank_train["day"].min()))

# Credit Card Default dataset.
credit_train = pd.read_csv("../train-test-datasets/credit-card-default/train_encoded.csv")
# Scaling the "SEX" and "PAY_" columns in Credit.
credit_train["SEX"] = ((credit_train["SEX"] - credit_train["SEX"].min()) / 
                       (credit_train["SEX"].max() - credit_train["SEX"].min()))
for p in [0, 2, 3, 4, 5, 6]:
    credit_train[f"PAY_{p}"] = ((credit_train[f"PAY_{p}"] - credit_train[f"PAY_{p}"].min()) / 
                                 (credit_train[f"PAY_{p}"].max() - credit_train[f"PAY_{p}"].min()))

def distance_metrics(real_dataset, synthetic_dataset, data_frac):
    """This function computes the distance-to-closest record (DCR) and nearest neighbor distance ratio (NNDR) metrics for a given real 
    and synthetic dataset. These datasets are both subsampled to a fraction of their size to aid computational complexity. 

    The code is based on the work of 
    
    Zhao, Z., Kunar, A., Birke, R. & Chen, L.Y. (2021). CTAB-GAN: Effective Table Data Synthesizing. 
    Proceedings of The 13th Asian Conference on Machine Learning, in Proceedings of Machine Learning Research 157:97-112 
    Available from https://proceedings.mlr.press/v157/zhao21a.html.

    whose code is available at:
    https://github.com/Team-TUD/CTAB-GAN/blob/main/model/eval/evaluation.py (accessed on March 19, 2026).

    real_dataset [array-like]: real dataset
    synthetic_dataset [array-like]: synthetic dataset
    data_frac [float]: subsampling ratio of real and synthetic datasets
    """

    # Subsampling real and synthetic datasets. 
    real_data = real_dataset.sample(int(real_dataset.shape[0]*data_frac), random_state = 42)
    synth_data = synthetic_dataset.sample(int(synthetic_dataset.shape[0]*data_frac), random_state = 42)

    # Computing the privacy metrics for the synthetic-synthetic and real-synthetic cases.
    try :
        ss_dist = pairwise_distances(synth_data)   # computing pairwise distances
        # Removing distances computed between the same entries, which would result in a distance of 0. 
        ss_clean = ss_dist[~np.eye(ss_dist.shape[0], dtype = bool)].reshape(ss_dist.shape[0], -1)
        ss_clean.sort()

        # Computing the privacy metrics for the synthetic-synthetic case. We only compute the metrics for the 'worst' 5th percentile distance results. 
        dcr_ss = np.quantile(ss_clean[:, 0], 0.05)
        nndr_ss = np.quantile(np.nan_to_num(ss_clean[:, 0]/ss_clean[:, 1]), 0.05)

        rs_dist = pairwise_distances(real_data, synth_data)   # computing pairwise distances
        rs_dist.sort()

        # Computing the privacy metrics for the real-synthetic case. We only compute the metrics for the 'worst' 5th percentile distance results. 
        dcr_rs = np.quantile(rs_dist[:, 0], 0.05)
        nndr_rs = np.quantile(np.nan_to_num(rs_dist[:, 0]/rs_dist[:, 1]), 0.05)
    except ValueError:
        dcr_ss = np.nan
        nndr_ss = np.nan

        dcr_rs = np.nan
        nndr_rs = np.nan

    return (dcr_ss, nndr_ss, dcr_rs, nndr_rs)

def results_analysis(file_path, epsi, real_dataset, data_frac, dataset):
    """This function computes the privacy metrics for a given GAN model, list of epsilon values and real dataset.

    If the GAN model does not have privacy constraints, set 'epsi' = [].
    
    file_path [string]: file path to synthetic datasets of a GAN model 
    epsi [list]: epsilon values
    real_dataset [array-like]: real dataset
    data_frac [float]: subsampling ratio of real and synthetic datasets
    dataset [string]: name of the dataset; can be set to "Adult", "Bank" or "Credit"
    """

    ss_results = {}   # synthetic-synthetic privacy results
    rs_results = {}   # real-synthetic privacy results
    runs = range(1, 6)  
    samples = range(1, 4)
    c = 1
    # Computing the privacy scores for the synthetic datasets of a GAN model.
    for r in runs:
        for s in samples:
            if epsi:
                synthetic_dataset = pd.read_csv(file_path + f"epsi_{epsi}/run{r}/sample{s}_encoded.csv")
            else:
                synthetic_dataset = pd.read_csv(file_path + f"run{r}/sample{s}_encoded.csv")

            # Scaling ordinal columns in the datasets to a range of [0, 1].
            if dataset == "Adult":
                synthetic_dataset["education-num"] = ((synthetic_dataset["education-num"] - synthetic_dataset["education-num"].min()) / 
                                                      (synthetic_dataset["education-num"].max() - synthetic_dataset["education-num"].min()))
            elif dataset == "Credit":
                synthetic_dataset["SEX"] = ((synthetic_dataset["SEX"] - synthetic_dataset["SEX"].min()) / 
                                            (synthetic_dataset["SEX"].max() - synthetic_dataset["SEX"].min()))
                for p in [0, 2, 3, 4, 5, 6]:
                    synthetic_dataset[f"PAY_{p}"] = ((synthetic_dataset[f"PAY_{p}"] - synthetic_dataset[f"PAY_{p}"].min()) / 
                                                     (synthetic_dataset[f"PAY_{p}"].max() - synthetic_dataset[f"PAY_{p}"].min()))
            elif dataset == "Bank":
                synthetic_dataset["day"] = ((synthetic_dataset["day"] - synthetic_dataset["day"].min()) / 
                                            (synthetic_dataset["day"].max() - synthetic_dataset["day"].min()))
            else:
                raise ValueError("Dataset not supported.")

            # Computing the privacy metrics for the synthetic dataset. 
            dcr_ss, nndr_ss, dcr_rs, nndr_rs = distance_metrics(real_dataset, synthetic_dataset, data_frac = data_frac)
            ss_results[f"sample{c}"] = (dcr_ss, nndr_ss)
            rs_results[f"sample{c}"] = (dcr_rs, nndr_rs)
            
            c += 1
            
    return pd.DataFrame(ss_results).rename(index = {0: "DCR", 1: "NNDR"}), pd.DataFrame(rs_results).rename(index = {0: "DCR", 1: "NNDR"})

def print_results(file_path, dataset, data_frac = 0.15):
    """Printing and saving the privacy scores of a GAN model.

    file_path [string]: file path to synthetic datasets of a GAN model
    dataset [string]: dataset name; can be set to "Adult", "Bank" or "Credit"
    data_frac [float]: subsampling ratio of real and synthetic datasets
    """
    
    if dataset == "Adult":
        real_dataset = adult_train
    elif dataset == "Bank":
        real_dataset = bank_train
    elif dataset == "Credit":
        real_dataset = credit_train
    else:
        raise ValueError("Dataset not supported.")

    # Computing the privacy results for each epsilon value.
    ss1_results, rs1_results = results_analysis(file_path = file_path, epsi = 1, real_dataset = real_dataset, data_frac = data_frac, dataset = dataset)
    ss2_results, rs2_results = results_analysis(file_path = file_path, epsi = 2, real_dataset = real_dataset, data_frac = data_frac, dataset = dataset)
    ss5_results, rs5_results = results_analysis(file_path = file_path, epsi = 5, real_dataset = real_dataset, data_frac = data_frac, dataset = dataset)
    ss8_results, rs8_results = results_analysis(file_path = file_path, epsi = 8, real_dataset = real_dataset, data_frac = data_frac, dataset = dataset)

    # Saving the real-synthetic privacy results as csv files.
    rs1_results.to_csv(file_path + f"priv-results_epsi-1.csv")
    rs2_results.to_csv(file_path + f"priv-results_epsi-2.csv")
    rs5_results.to_csv(file_path + f"priv-results_epsi-5.csv")
    rs8_results.to_csv(file_path + f"priv-results_epsi-8.csv")

    # Printing the average privacy scores for the synthetic-synthetic case.
    print("DCR results (Synthetic)")
    print(np.nanmean(ss1_results.iloc[0]), np.nanstd(ss1_results.iloc[0]))
    print(np.nanmean(ss2_results.iloc[0]), np.nanstd(ss2_results.iloc[0]))
    print(np.nanmean(ss5_results.iloc[0]), np.nanstd(ss5_results.iloc[0]))
    print(np.nanmean(ss8_results.iloc[0]), np.nanstd(ss8_results.iloc[0]))
    print()
    print("NNDR results (Synthetic)")
    print(np.nanmean(ss1_results.iloc[1]), np.nanstd(ss1_results.iloc[1]))
    print(np.nanmean(ss2_results.iloc[1]), np.nanstd(ss2_results.iloc[1]))
    print(np.nanmean(ss5_results.iloc[1]), np.nanstd(ss5_results.iloc[1]))
    print(np.nanmean(ss8_results.iloc[1]), np.nanstd(ss8_results.iloc[1]))
    print()

    # Printing the average privacy scores for the real-synthetic case.
    print("DCR results (Real-Synthetic)")
    print(np.nanmean(rs1_results.iloc[0]), np.nanstd(rs1_results.iloc[0]))
    print(np.nanmean(rs2_results.iloc[0]), np.nanstd(rs2_results.iloc[0]))
    print(np.nanmean(rs5_results.iloc[0]), np.nanstd(rs5_results.iloc[0]))
    print(np.nanmean(rs8_results.iloc[0]), np.nanstd(rs8_results.iloc[0]))
    print()
    print("NNDR results (Real-Synthetic)")
    print(np.nanmean(rs1_results.iloc[1]), np.nanstd(rs1_results.iloc[1]))
    print(np.nanmean(rs2_results.iloc[1]), np.nanstd(rs2_results.iloc[1]))
    print(np.nanmean(rs5_results.iloc[1]), np.nanstd(rs5_results.iloc[1]))
    print(np.nanmean(rs8_results.iloc[1]), np.nanstd(rs8_results.iloc[1]))

## Adult Dataset (Real)

In [ ]:
# DCR and NNDR of Adult dataset.
results = distance_metrics(real_dataset = adult_train, synthetic_dataset = adult_train, data_frac = 0.15)
print("Synthetic (DCR, NNDR):", results[:2])
print("Real-Synthetic (DCR, NNDR):", results[2:])

## Synthetic Adult Dataset (TabFairGAN)

In [ ]:
# DCR and NNDR of Adult dataset (TabFairGAN).
file_path = "../synthetic-datasets_TabFair/adult/"
ss_results, rs_results = results_analysis(file_path = file_path, epsi = [], real_dataset = adult_train, data_frac = 0.15, dataset = "Adult")
print("Synthetic (DCR, NNDR):", (np.nanmean(ss_results.iloc[0]), np.nanstd(ss_results.iloc[0])), 
                                (np.nanmean(ss_results.iloc[1]), np.nanstd(ss_results.iloc[1])))
print("Real-Synthetic (DCR, NNDR):", (np.nanmean(rs_results.iloc[0]), np.nanstd(rs_results.iloc[0])), 
                                     (np.nanmean(rs_results.iloc[1]), np.nanstd(rs_results.iloc[1])))

# Saving the privacy results as a csv file.
rs_results.to_csv(file_path + f"priv-results.csv")

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (DP-CTGAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DPCTGAN_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GANs with Demographic Parity Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dem)_B=512/adult/"
print_results(file_path, dataset = "Adult")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_dem)_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GANs with Disparate Impact Loss)

In [ ]:
# Assigning file path. 
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/adult/"
print_results(file_path, dataset = "Adult")

In [ ]:
# Assigning file path. 
file_path = "../synthetic-datasets_FairDP-GAN(clf_dis)_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GAN with Equal Opportunity Loss)

In [ ]:
# Assigning file path. 
file_path = "../synthetic-datasets_FairDP-GAN(clf_eqopp)_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Synthetic Adult Dataset (Fair DP-GANs with All Fair Losses)

In [ ]:
# Assigning file path. 
file_path = "../synthetic-datasets_FairDP-GAN(dem-dis)_B=512/adult/"
print_results(file_path, dataset = "Adult")

In [ ]:
# Assigning file path. 
file_path = "../synthetic-datasets_FairDP-GAN(clf_all)_B=512/adult/"
print_results(file_path, dataset = "Adult")

## Bank Marketing Dataset (Real)

In [ ]:
# DCR and NNDR of Bank Marketing dataset.
results = distance_metrics(real_dataset = bank_train, synthetic_dataset = bank_train, data_frac = 0.15)
print("Synthetic (DCR, NNDR):", results[:2])
print("Real-Synthetic (DCR, NNDR):", results[2:])

## Synthetic Bank Marketing Dataset (TabFairGAN)

In [ ]:
# DCR and NNDR of Bank Marketing dataset (TabFairGAN).
file_path = "../synthetic-datasets_TabFair/bank-marketing/"
ss_results, rs_results = results_analysis(file_path = file_path, epsi = [], real_dataset = bank_train, data_frac = 0.15, dataset = "Bank")
print("Synthetic (DCR, NNDR):", (np.nanmean(ss_results.iloc[0]), np.nanstd(ss_results.iloc[0])), 
                                (np.nanmean(ss_results.iloc[1]), np.nanstd(ss_results.iloc[1])))
print("Real-Synthetic (DCR, NNDR):", (np.nanmean(rs_results.iloc[0]), np.nanstd(rs_results.iloc[0])), 
                                     (np.nanmean(rs_results.iloc[1]), np.nanstd(rs_results.iloc[1])))

# Saving the privacy results as a csv file.
rs_results.to_csv(file_path + f"priv-results.csv")

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (DP-CTGAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DPCTGAN_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GANs with Demographic Parity Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dem)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_dem)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GANs with Disparate Impact Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_dis)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GAN with Equal Opportunity Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_eqopp)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Synthetic Bank Marketing Dataset (Fair DP-GANs with All Fair Losses)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dem-dis)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_all)_B=512/bank-marketing/"
print_results(file_path, dataset = "Bank")

## Credit Card Default Dataset (Real)

In [ ]:
# DCR and NNDR of Credit Card Default dataset.
results = distance_metrics(real_dataset = credit_train, synthetic_dataset = credit_train, data_frac = 0.15)
print("Synthetic-Synthetic (DCR, NNDR):", results[:2])
print("Real-Synthetic (DCR, NNDR):", results[2:])

## Synthetic Credit Card Default Dataset (TabFairGAN)

In [ ]:
# DCR and NNDR of Credit Card Default dataset (TabFairGAN).
file_path = "../synthetic-datasets_TabFair/credit-card-default/"
ss_results, rs_results = results_analysis(file_path = file_path, epsi = [], real_dataset = credit_train, data_frac = 0.15, dataset = "Credit")
print("Synthetic (DCR, NNDR):", (np.nanmean(ss_results.iloc[0]), np.nanstd(ss_results.iloc[0])), 
                                (np.nanmean(ss_results.iloc[1]), np.nanstd(ss_results.iloc[1])))
print("Real-Synthetic (DCR, NNDR):", (np.nanmean(rs_results.iloc[0]), np.nanstd(rs_results.iloc[0])), 
                                     (np.nanmean(rs_results.iloc[1]), np.nanstd(rs_results.iloc[1])))

# Saving the privacy results as a csv file.
rs_results.to_csv(file_path + f"priv-results.csv")

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DP-GAN_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (DP-CTGAN)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_DPCTGAN_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GANs with Demographic Parity Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dem)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_dem)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GANs with Disparate Impact Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dis)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_dis)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GAN with Equal Opportunity Loss)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_eqopp)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

## Synthetic Credit Card Default Dataset (Fair DP-GANs with All Fair Losses)

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(dem-dis)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")

In [ ]:
# Assigning file path.
file_path = "../synthetic-datasets_FairDP-GAN(clf_all)_B=512/credit-card-default/"
print_results(file_path, dataset = "Credit")